# Context: application data passed to the agent

In the previous lesson, `state` stored the messages and the agent's result. Here, the application knows additional data about the **current invocation**: the user ID and preferred language. We will pass these via `context=...`, separate from the content of the question. Logging and the database are simulated: there are no passwords, servers, SQL connections, or account operations.

At the end, we will see two uses for this same data: the application code will prepare a query plan for the database, while the agent's tool will read the profile from a local dictionary. The model will only see the information that is intentionally included in the prompt or the tool's output.

[Runtime context in LangChain](https://docs.langchain.com/oss/python/langchain/tools#context) · [Agent invocation](https://docs.langchain.com/oss/python/langchain/agents#invocation)


In [ ]:
# All imports and minimum project path setup are in this cell.
import sys
from dataclasses import dataclass
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, dynamic_prompt
from langchain.tools import ToolRuntime, tool
from model_config import get_model


## 1. Where does `user_id` come from?

In a real application, the server establishes identity after verifying the session/login and only then creates the context. The dictionaries below **simulate the result of this step**. They are not a login mechanism. We do not take the `user_id` from the user's query or from an argument invented by the model.

`RequestContext` describes the data available during a single call. We change the context when handling a different user; we do not need to change the query text or create a new agent.


In [ ]:
@dataclass(frozen=True)
class RequestContext:
    user_id: str
    language: str


# Mock results of previously verified sessions; no authentication is performed here.
MOCK_SESSIONS = [
    {"subject": "u-101", "language": "en-US"},
    {"subject": "u-202", "language": "en-GB"},
]
contexts = [
    RequestContext(user_id=session["subject"], language=session["language"])
    for session in MOCK_SESSIONS
]
print("Contexts created by the application:", contexts)


## 2. The same context can be used by the application code

The function below **only shows the plan** of the query. `user_id` is a separate parameter, not part of the concatenated SQL text. There is no database connection or `execute()`. In a functioning system, read permissions would have to be enforced by the application/database layer.


In [ ]:
def preview_profile_query(context: RequestContext) -> dict:
    return {
        "sql": "SELECT display_name, plan FROM profiles WHERE user_id = ?",
        "params": (context.user_id,),
    }


print("Plan for the first session:", preview_profile_query(contexts[0]))
print("Plan for the second session:", preview_profile_query(contexts[1]))


## 3. The tool reads `runtime.context`

Instead of a real database, we use two dummy records in a dictionary. The `my_profile` tool **does not have a `user_id` argument in the schema visible to the model**. When the agent calls it, LangChain will inject `ToolRuntime`; the tool's code will retrieve the ID from the context provided by the application. The model will only see the returned name and plan, not the raw ID.


In [ ]:
DEMO_PROFILES = {
    "u-101": {"display_name": "Ala", "plan": "Plus"},
    "u-202": {"display_name": "Bartek", "plan": "Basic"},
}


@tool
def my_profile(runtime: ToolRuntime[RequestContext]) -> str:

    """Read the name and plan of the current user from the demo data."""
    profile = DEMO_PROFILES.get(runtime.context.user_id)

    if profile is None:
        return "No profile found in the demo data."
    return f"Name: {profile['display_name']}; plan: {profile['plan']}"


## 4. What from the context do we show to the model?

The `context=...` itself does not append data to the prompt. Here, `dynamic_prompt` deliberately passes **only the preferred language**. The ID remains in the application and tool code. We save the prompt to a list solely so it is visible what the model actually received. This list also works only in the current kernel.


In [ ]:
PROMPTS_SHOWN_TO_MODEL = []


@dynamic_prompt
def language_prompt(request: ModelRequest) -> str:
    language = "in American English" if request.runtime.context.language == "en-US" else "in British English"
    base_prompt = request.system_message.text if request.system_message else ""
    prompt = f"{base_prompt}\nRespond {language}."
    PROMPTS_SHOWN_TO_MODEL.append(prompt)
    return prompt


model = get_model()
agent = create_agent(
    model=model,
    tools=[my_profile],
    context_schema=RequestContext,
    system_prompt="When the user asks about their plan, check the data, never guess it. Of course, respond in the user's language.",
    middleware=[language_prompt],
)


## 5. The same question, two different contexts

The question does not contain an ID or a plan name. Each `agent.invoke(...)` receives a separate `context`. The agent may request `my_profile`, Python reads the correct demonstration record, and the model formulates an answer based on the tool's result. We display tool messages separately to avoid mixing the model's decisions with what Python returned.


In [ ]:
QUESTION = "What is my plan? Answer briefly."


def ask_about_plan(context: RequestContext) -> dict:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": QUESTION}]},
        context=context,
    )
    print("Application context:", context)
    for message in result["messages"]:
        if message.type == "ai" and message.tool_calls:
            print("Model requested:", [call["name"] for call in message.tool_calls])
        elif message.type == "tool":
            print("Python returned from tool:", message.content)
    print("Model response:", result["messages"][-1].content, "\n\n")
    return result


first_result = ask_about_plan(contexts[0])
second_result = ask_about_plan(contexts[1])


## What just happened?

| Data | Who sets it? | Who sees it in this example? |
|---|---|---|
| Question `What is my plan?` | user | model |
| `user_id` | fictional application session | application code and the tool through `runtime.context` |
| `language` | fictional application session | application code; the model only after it is deliberately added to the system prompt |
| Name and plan | local dictionary standing in for a data source | tool, then model through the tool result |

`context` is data for the current invocation, available to tools and middleware. `state` from the previous lesson is agent run data, such as `messages` and `structured_response`. The `thread_id` from the memory lesson identifies a conversation saved by a checkpointer. This notebook has no checkpointer, real login, or database connection.

The key boundary: the model can **request** a tool and describe its result, but it does not choose whose ID the tool uses. In a real system, the application must verify the session and permissions before passing context.
